# Full Pipeline — Playground

End-to-end run of the **entire stock-selection pipeline** in one notebook, top to bottom.
Each stage below has a short explainer of *what* it does and *which dials* (in
`backend/config.py`) govern it.

```
Stage 1  Universe Filter   weekly   ~60-80 large-cap liquid stocks  → watchlist.csv
Stage 2  Momentum Scanner  nightly  scores the watchlist 0-3        → top candidates
Gate 1   Hard Threat       rules    blocks on macro/market shocks   (no Claude)
Gate 2   News Threat       Claude   blocks on catastrophic news
Gate 3   Sentiment         Claude   direction + confidence          → pass/block/caution
Gate 4   Contradiction     Claude   stock-vs-market divergence
Gate 5   Edge / EV         rules    win-probability + expected value → BUY / SKIP
```

The funnel narrows at every step: a few dozen universe names → a handful of momentum
candidates → the gates filter those down to the final **BUY** list.

> **⚠️ Live API + Claude calls.** Gates 2-4 each make one Claude (Haiku) call, so
> ~3 calls per surviving candidate. Needs `.env` keys: `ANTHROPIC_API_KEY` plus a news source
> (`ALPACA_API_KEY` + `ALPACA_SECRET_KEY`, or `NEWS_API_KEY`); the earnings/universe steps use
> `FINNHUB_API_KEY`. Missing keys degrade gracefully — gates **block** rather than crash.

> **Every tunable number lives in `backend/config.py`** — the single dial board. This notebook
> only *reads* those values; tweak the strategy there.

## Setup — imports & path wiring

The numbered folders (`01_scanner`, `02_intelligence`) are **not** importable Python packages,
so — exactly like every other playground — we add the relevant directories to `sys.path` and
import each module by its bare name. We also add `backend/` itself so `from config import ...`
works here too.

Both `universe_filter` and `momentum_scanner` define a **cwd-relative** `WATCHLIST_PATH`
(`'data/watchlist.csv'`). We re-point both at the absolute path so the notebook runs no matter
where the kernel was launched.

In [2]:
import sys
import pathlib
import pandas as pd

# --- Anchor to backend/ regardless of where the kernel started -------------
nb_dir = pathlib.Path('.').resolve()
if (nb_dir / 'config.py').exists():
    backend_dir = nb_dir                                  # launched from backend/
else:
    backend_dir = pathlib.Path('backend').resolve()       # launched from repo root

intelligence_dir = backend_dir / '02_intelligence'
scanner_dir      = backend_dir / '01_scanner'

# Bare-name imports need each module's own dir on sys.path.
for p in [
    backend_dir,                                  # → config.py
    scanner_dir,                                  # → universe_filter, momentum_scanner
    intelligence_dir,                             # → constants, helpers/*
    intelligence_dir / 'gate1_hard_threat',
    intelligence_dir / 'gate2_news_threat',
    intelligence_dir / 'gate3_sentiment',
    intelligence_dir / 'gate4_contradiction',
    intelligence_dir / 'gate5_signal',
]:
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- Stage 1 & 2: scanner --------------------------------------------------
import universe_filter
import momentum_scanner
from universe_filter import run_universe_filter
from momentum_scanner import run_scan

# Re-point both cwd-relative watchlist paths at the real file.
WATCHLIST_PATH = str(scanner_dir / 'data' / 'watchlist.csv')
universe_filter.WATCHLIST_PATH = WATCHLIST_PATH
momentum_scanner.WATCHLIST_PATH = WATCHLIST_PATH

# --- Gates 1-5 -------------------------------------------------------------
from hard_threat_gate1 import get_shared_market_data, screen_gate1_hard_threats
from news_threat_gate2 import assess_gate2_news_threat
from sentiment_gate3 import evaluate_gate3_sentiment
from contradiction_gate4 import detect_gate4_contradiction
from signal_gate5 import decide_gate5_signal

# --- Fetchers shared across gates ------------------------------------------
from helpers.fetchers.news import fetch_news
from helpers.fetchers.market import get_market_context

# --- Active dials (read-only echo) -----------------------------------------
import config
print('Active strategy dials (edit in backend/config.py):')
print(f"  Universe : MIN_MARKET_CAP=${config.MIN_MARKET_CAP:,.0f}  MIN_PRICE=${config.MIN_PRICE}  "
      f"ATR%=[{config.MIN_ATR_PCT}, {config.MAX_ATR_PCT}]")
print(f"  Scanner  : MIN_SCORE={config.MIN_SCORE}  TOP_N={config.TOP_N}  "
      f"RSI=[{config.RSI_MIN}, {config.RSI_MAX}]")
print(f"  Gate 3   : MIN_CONFIDENCE={config.MIN_CONFIDENCE}")
print(f"  Gate 5   : MIN_EDGE_PCT={config.MIN_EDGE_PCT:.0%}  WIN_PROB_BASE={config.WIN_PROB_BASE:.0%}")
print(f"  Watchlist: {WATCHLIST_PATH}")

Active strategy dials (edit in backend/config.py):
  Universe : MIN_MARKET_CAP=$100,000,000,000  MIN_PRICE=$10.0  ATR%=[1.0, 5.0]
  Scanner  : MIN_SCORE=2  TOP_N=15  RSI=[50, 70]
  Gate 3   : MIN_CONFIDENCE=6
  Gate 5   : MIN_EDGE_PCT=4%  WIN_PROB_BASE=35%
  Watchlist: /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv


## Stage 1 — Universe Filter  *(weekly screen → `watchlist.csv`)*

The Tier-1 screen defines **which stocks are even allowed onto the watchlist**. It asks
TradingView for US large-caps and keeps only the liquid, tradable ones, then drops names with
earnings in the next few days. The survivors (with `price, volume, atr, rsi, sma20, sma50,
sector`) are written to `watchlist.csv`.

Dials (`config.py`): `MIN_MARKET_CAP` (≥ \$100B), `MIN_AVG_VOLUME` (≥ 1M shares/day),
`MIN_PRICE` (≥ \$10), `MIN_ATR_PCT`/`MAX_ATR_PCT` (1–5% daily range), `EARNINGS_WINDOW_DAYS`
(exclude names reporting within 5 days), `SCREENER_LIMIT`.

This is a **weekly** job and a live regeneration overwrites the existing watchlist (TradingView
+ Finnhub calls). So it is **opt-in**: by default we just read the current `watchlist.csv` and
report what's in it. Flip `RUN_UNIVERSE_FILTER = True` to rebuild it live.

In [3]:
RUN_UNIVERSE_FILTER = True   # set True to rebuild watchlist.csv live (slow; overwrites!)

if RUN_UNIVERSE_FILTER:
    # max_price passed explicitly to skip the Alpaca portfolio lookup (reproducible runs).
    count = run_universe_filter(max_price=400.0)
    print(f'[universe] regenerated watchlist with {count} stocks')

universe_df = pd.read_csv(WATCHLIST_PATH)
print(f'Universe: {len(universe_df)} stocks in watchlist.csv\n')

# Sector breakdown — how the universe is spread across the market.
print('By sector:')
print(universe_df['sector'].value_counts().to_string())

print('\nSample rows:')
universe_df.head(10)

[universe] price ceiling: $400.00
[universe] querying TradingView screener...
[universe] 71 stocks after ATR% filter (1.0%–5.0%)
[universe] saved 71 tickers to /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv
[universe] regenerated watchlist with 71 stocks
Universe: 71 stocks in watchlist.csv

By sector:
sector
Finance                  16
Electronic Technology    11
Health Technology        10
Technology Services       8
Retail Trade              6
Consumer Non-Durables     5
Consumer Services         4
Communications            3
Energy Minerals           3
Transportation            2
Utilities                 2
Non-Energy Minerals       1

Sample rows:


,ticker,price,volume,atr,atr_pct,rsi,sma20,sma50,sector
0,NVDA,199.61,153527722,7.08,3.55,44.8,204.58,209.94,Electronic Technology
1,AMZN,243.82,82516932,8.41,3.45,49.5,240.72,255.60,Retail Trade
2,AAPL,294.79,80988785,8.20,2.78,51.3,294.90,292.68,Electronic Technology
3,T,20.45,78156557,0.68,3.34,27.1,22.52,24.20,Communications
4,MSFT,387.44,61227272,13.42,3.46,48.3,388.96,408.33,Technology Services
5,PFE,23.84,60605915,0.57,2.37,32.3,25.18,25.77,Health Technology
6,NFLX,73.98,56585669,2.51,3.39,36.9,77.45,84.41,Technology Services
7,BAC,58.30,38281416,1.18,2.03,68.4,56.09,53.54,Finance
8,VZ,41.72,37357039,1.33,3.19,29.7,45.69,46.72,Communications
9,AVGO,371.81,29769844,17.91,4.82,42.7,390.38,410.31,Electronic Technology


## Stage 2 — Momentum Scanner  *(nightly → top candidates)*

The Tier-2 scan scores every watchlist stock **0–3**, one point each for:

1. RSI inside the momentum zone `[RSI_MIN, RSI_MAX]` — rising but not yet overbought,
2. price **>** SMA20 — short-term uptrend,
3. SMA20 **>** SMA50 — golden-cross alignment.

It keeps names scoring ≥ `MIN_SCORE` and returns the strongest `TOP_N` by score.

Dials (`config.py`): `MIN_SCORE` (=2), `TOP_N` (=15), `RSI_MIN`/`RSI_MAX` (50–70).

Note `run_scan()` drops the `sector` column, but Gates 1 & 4 need it — so we keep a
`sector_by_ticker` lookup from the watchlist for later.

In [4]:
sector_by_ticker = (
    universe_df.dropna(subset=['sector'])
    .set_index('ticker')['sector']
    .to_dict()
)

candidates_df = run_scan()   # uses MIN_SCORE / TOP_N from config

if candidates_df is None or candidates_df.empty:
    print('No candidates from run_scan — check watchlist.csv')
else:
    print(f'\nFunnel so far: {len(universe_df)} universe '
          f'→ {len(candidates_df)} candidates (score ≥ {config.MIN_SCORE}, top {config.TOP_N})')

candidates_df

[scanner] loaded 71 stocks from /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv
[scanner] 41 stocks with score >= 2
[scanner] returning top 15 candidates

Funnel so far: 71 universe → 15 candidates (score ≥ 2, top 15)


,ticker,price,score,atr,rsi,sma20,sma50
0,APH,172.66,3,7.01,66.5,158.03,145.85
1,PGR,224.67,3,5.74,67.8,208.33,202.78
2,HWM,267.30,3,9.29,51.5,265.97,259.22
3,BNY,146.40,3,3.40,59.1,143.78,139.02
4,IBKR,93.02,3,3.64,57.8,90.44,85.84
5,RTX,190.83,3,4.43,60.1,184.13,179.50
6,DHR,193.68,3,5.35,62.8,184.76,178.61
7,HD,352.54,3,8.28,67.7,330.53,323.06
8,WELL,231.52,3,5.62,68.7,213.89,213.06
9,AXP,348.57,3,7.84,69.8,329.76,321.40


## Run-level inputs

Before the gate loop we set the **portfolio context** Gate 1 needs (its loss-limit and
position-sizing checks), and fetch the **market-wide data once** (`get_shared_market_data()` →
VIX / SPY / hours-to-next-macro) so every Gate 1 call reuses it instead of re-hitting the API.

`TOP_N` here caps how many candidates we actually push through the (paid) Claude gates.

In [5]:
portfolio_value = 100_000.0   # total account value (Gate 1 loss-limit / sizing)
daily_pnl       = 0.0          # today's realised + unrealised P&L (negative = loss)
TOP_N           = 10          # cap candidates pushed through the Claude gates

shared = get_shared_market_data()   # VIX / SPY / macro — fetched once for all Gate 1 calls
print('Shared market data:', shared)

Shared market data: {'vix': {'level': 16.26, 'change_pct_today': -0.0116, 'prior_close': 16.45}, 'spy': {'price': 747.51, 'change_pct_today': 0.001, 'prior_close': 746.77}, 'macro_hours': 17.1}


## Gates 1–5 — the decision chain

Each candidate runs the gates **in order, stopping at the first failure** (fail fast keeps
Claude cost down). The chain:

- **Gate 1 — Hard Threat** *(rules)*: blocks on macro/market shocks — VIX spike, SPY/sector
  selloff, pre-market gap, imminent macro event, earnings tomorrow, fresh 8-K, daily loss
  limit. Thresholds in `config.BLOCK_THRESHOLDS`.
- **Gate 2 — News Threat** *(Claude)*: reads the headlines, blocks on a catastrophic story
  (fraud, recall, regulatory action, …). News is fetched **once** here and reused by Gate 3.
- **Gate 3 — Sentiment** *(Claude)*: returns direction + confidence (0–10). `MIN_CONFIDENCE`
  turns that into pass / block / pass-with-caution.
- **Gate 4 — Contradiction** *(Claude)*: blocks when the market backdrop contradicts a bullish
  entry (stock up while sector/market is risk-off). Uses `get_market_context(sector)`.
- **Gate 5 — Edge / EV** *(rules)*: maps momentum score + Gate 3 sentiment to a win
  probability, computes expected value, and issues **BUY** if `EV ≥ MIN_EDGE_PCT` else
  **SKIP**. Also returns the trade levels (entry/stop/target).

The driver below records, for every ticker, where it stopped and the Gate 3/Gate 5 numbers so
the results table tells the whole story.

In [6]:
def _row(ticker, decision, g3=None, g5=None):
    """One results-table row; gate3/gate5 fields filled only when those gates ran."""
    return {
        'ticker': ticker,
        'final_decision': decision,
        'g3_direction': g3.get('direction') if g3 else None,
        'g3_confidence': g3.get('confidence') if g3 else None,
        'ev': round(g5['expected_value'], 3) if g5 else None,
        'win_prob': round(g5['win_probability'], 3) if g5 else None,
        'position_confidence': g5['position_confidence'] if g5 else None,
    }

rows, processed = [], 0

if candidates_df is not None and not candidates_df.empty:
    for _, r in candidates_df.iterrows():
        if processed >= TOP_N:
            break
        ticker = str(r['ticker'])
        sector = sector_by_ticker.get(ticker)
        if sector is None:
            print(f'{ticker:<5} skipped — no sector in watchlist')
            continue
        processed += 1

        candidate = {
            'ticker': ticker,
            'company_name': ticker,
            'sector': sector,
            'price': float(r['price']),
            'atr': float(r['atr']),
            'score': int(r['score']),
        }

        # Gate 1 — hard threats (rules, reuses shared market data)
        g1 = screen_gate1_hard_threats(candidate, shared, portfolio_value, daily_pnl)
        if not g1['passed']:
            rows.append(_row(ticker, f"BLOCKED_G1:{g1.get('block_reason')}"))
            continue

        # Gates 2 & 3 share a single news fetch
        headlines = fetch_news(ticker) or []
        g2 = assess_gate2_news_threat(candidate, headlines)
        if not g2['passed']:
            rows.append(_row(ticker, 'BLOCKED_G2'))
            continue
        g3 = evaluate_gate3_sentiment(candidate, headlines)
        if not g3['passed']:
            rows.append(_row(ticker, 'BLOCKED_G3', g3=g3))
            continue

        # Gate 4 — contradiction vs the live market backdrop
        market_context = get_market_context(sector)
        if market_context is None:
            rows.append(_row(ticker, 'BLOCKED_G4:no_market_context', g3=g3))
            continue
        g4 = detect_gate4_contradiction(candidate, g3, market_context)
        if not g4['passed']:
            rows.append(_row(ticker, f"BLOCKED_G4:{g4.get('action')}", g3=g3))
            continue

        # Gate 5 — edge check + EV
        g5 = decide_gate5_signal(candidate, {'gate1': g1, 'gate2': g2, 'gate3': g3, 'gate4': g4})
        rows.append(_row(ticker, g5['decision'], g3=g3, g5=g5))

print(f'\nProcessed {processed} candidates through the gates.')

[gate1] APH: BLOCKED — sector
[gate1] PGR: passed all 8 checks
[gate2] PGR: passed — no threat across 2 headlines
[gate3] PGR: passed (caution) — NEUTRAL conf=6
[gate4] PGR: passed — no contradiction (risk=NONE)
[gate5] PGR: BUY — EV 0.110 | win_prob=37% | LOW
[gate1] HWM: BLOCKED — sector
[gate1] BNY: passed all 8 checks
[gate2] BNY: passed — no threat across 5 headlines
[gate3] BNY: passed — BULLISH conf=8
[gate4] BNY: passed — no contradiction (risk=NONE)
[gate5] BNY: BUY — EV 0.875 | win_prob=62% | HIGH
[gate1] IBKR: BLOCKED — premarket_gap
[gate1] RTX: BLOCKED — sector
[gate1] DHR: passed all 8 checks
[gate2] DHR: passed — no threat across 2 headlines
[gate3] DHR: BLOCKED — NEUTRAL conf=0: These headlines are completely unrelated to DHR (Danaher Corporation) and contain no financial, business, or stock-relevant information whatsoever.
[gate1] HD: passed all 8 checks
[gate2] HD: passed — no threat across 1 headlines
[gate3] HD: BLOCKED — NEUTRAL conf=4: While the headline highlight

## Results — the funnel & the BUY list

The table below shows every processed ticker, where it stopped, and the Gate 3 / Gate 5
numbers. Then the funnel counts (`universe → scanned → processed → BUY`) and, for each **BUY**,
the trade levels Gate 5 produced (entry / stop / target).

In [6]:
results_df = pd.DataFrame(rows)

if results_df.empty:
    print('No candidates were processed.')
else:
    print(results_df.to_string(index=False))

    buys = results_df[results_df['final_decision'] == 'BUY']
    print(f'\nFunnel: {len(universe_df)} universe '
          f'→ {len(candidates_df)} scanned '
          f'→ {processed} processed '
          f'→ {len(buys)} BUY')

    # Trade levels for each BUY (recompute via Gate 5 to pull entry/stop/target).
    if not buys.empty:
        print('\nBUY trade levels:')
        for _, b in buys.iterrows():
            t = str(b['ticker'])
            row = candidates_df[candidates_df['ticker'] == t].iloc[0]
            cand = {'ticker': t, 'price': float(row['price']),
                    'atr': float(row['atr']), 'score': int(row['score'])}
            g5 = decide_gate5_signal(cand, {
                'gate3': {'passed': True,
                          'direction': b['g3_direction'],
                          'confidence': int(b['g3_confidence']),
                          'caution': False}})
            lv = g5['trade_levels']
            print(f"  {t:<5} entry={lv['entry']:.2f}  stop={lv['stop']:.2f}  "
                  f"target={lv['target']:.2f}  R:R={lv['reward_risk']:.1f}  "
                  f"EV={b['ev']}  conf={b['position_confidence']}")

ticker             final_decision g3_direction  g3_confidence    ev  win_prob position_confidence
  BKNG                 BLOCKED_G3      BEARISH            6.0   NaN       NaN                None
    PG BLOCKED_G4:FLAG_FOR_REVIEW      BULLISH            6.0   NaN       NaN                None
   HON   BLOCKED_G1:premarket_gap         None            NaN   NaN       NaN                None
   RTX                        BUY      BULLISH            7.0 0.688     0.562                HIGH
  FTNT                 BLOCKED_G3      BEARISH            7.0   NaN       NaN                None
   DHR                 BLOCKED_G3      NEUTRAL            5.0   NaN       NaN                None
   HWM                        BUY      BULLISH            7.0 0.688     0.562                HIGH
   APH   BLOCKED_G1:premarket_gap         None            NaN   NaN       NaN                None
    MO                 BLOCKED_G3      NEUTRAL            5.0   NaN       NaN                None
     V BLOCKED_G4:FL

## Free-play

Scratch cell — tweak `portfolio_value`, `TOP_N`, or a single candidate and re-run pieces above.